未选取特定类别的图片：

![1.png](1.png)

所有特定的类别显示在同一张图片里，如：

![2.png](2.png)

In [ ]:
# 通过jpg图片和标注的json文件得到png图片
# pip install labelme==3.16.7

import base64
import json
import os
import os.path as osp
import numpy as np
import PIL.Image
from labelme import utils

if __name__ == '__main__':
    jpgs_path   = "JPEGImages_test"
    pngs_path   = "SegmentationClass_part" # 生成的 png 图像存放路径
    classes = ["_background_", "egg", "light", "white"]
    specific_classes = ["_background_", "white"]
    # specific_classes = ["_background_","egg", "light"]  # 指定要生成的类别
    count = os.listdir("before_test") 
    os.makedirs(pngs_path, exist_ok=True)
    
    for i in range(0, len(count)):
        path = os.path.join("before_test", count[i])
        if os.path.isfile(path) and path.endswith('json'):
            data = json.load(open(path))
            
            if data['imageData']:
                imageData = data['imageData']
            else:
                imagePath = os.path.join(os.path.dirname(path), data['imagePath'])
                with open(imagePath, 'rb') as f:
                    imageData = f.read()
                    imageData = base64.b64encode(imageData).decode('utf-8')
            img = utils.img_b64_to_arr(imageData)
            label_name_to_value = {'_background_': 0}
            

            for shape in data['shapes']:
                label_name = shape['label']
                print('label_name:', label_name) 
                if label_name not in specific_classes:
                    continue
                if label_name in label_name_to_value:
                    label_value = label_name_to_value[label_name]
                else:
                    label_value = len(label_name_to_value)
                    label_name_to_value[label_name] = label_value
            
            
            # label_values must be dense
            label_values, label_names = [], []
            for ln, lv in sorted(label_name_to_value.items(), key=lambda x: x[1]):
                label_values.append(lv)
                label_names.append(ln)
            print('label_values:', label_values)
            print('label_names:', label_names)
            print('list(range(len(label_values)))', list(range(len(label_values))))
            assert label_values == list(range(len(label_values)))
            print('label_name_to_value',label_name_to_value)

            ######################################################################
            # 过滤出指定类别的 shapes
            filtered_shapes = [shape for shape in data['shapes'] if shape['label'] in specific_classes]

            lbl = utils.shapes_to_label(img.shape, filtered_shapes, label_name_to_value)
            # lbl = utils.shapes_to_label(img.shape, data['shapes'], label_name_to_value)
            
            PIL.Image.fromarray(img).save(osp.join(jpgs_path, count[i].split(".")[0] + '.jpg'))

            # 将图像中的每个像素值替换为对应的标签类别索引
            new = np.zeros([np.shape(img)[0], np.shape(img)[1]])

            for name in label_names:
                index_json = label_names.index(name)
                print('index_json', index_json)
                index_all = classes.index(name)
                print('index_all', index_all)
                # 创建一个布尔数组，其中值为 True 的位置表示图像 lbl 中的像素值等于 index_json。布尔值会被转换为 0 和 1，其中 True 对应 1，False 对应 0。
                # 将 index_all 乘以该布尔数组，得到一个与 index_json 对应的标签索引。例如，如果 index_json 是 2，index_all 是 3，那么该区域的值将被设置为 3。
                # new = new + index_all * (np.array(lbl) == index_json)

                '''
                    使用 index_json 替换 index_all，这样所提取到的物体的索引就不会保持在原classes列表中的
                    比如 classes = ["_background_", "egg", "light", "white"]
                    选取 specific_classes = ["_background_", "white"]
                    那么 new 中的像素值将会是 0 和 1，而不是 0 和 3
                '''
                new = new + index_json * (np.array(lbl) == index_json)
            
            print(osp.join(pngs_path, count[i].split(".")[0] + '.png'))
            utils.lblsave(osp.join(pngs_path, count[i].split(".")[0] + '.png'), new)
            print('Saved ' + count[i].split(".")[0] + '.jpg and ' + count[i].split(".")[0] + '.png')


选定的类别分别生成单独的图片